# 当前版本沪深全市场回归与 profiling
第一阶段七个既有日期、两市股票和 ETF。只有十四个日市场任务全部完成、无不匹配/数据错误/来源缺失并通过审计后，才随机选三个其他完整且无既往运行记录的日期。六个独立进程，固定二进制及源代码快照。停牌排除不算匹配。

In [ ]:
from pathlib import Path
import json
root = Path.cwd() if (Path.cwd() / 'Cargo.toml').exists() else Path.cwd().parent
out = root / 'reports/20260907-current-full-regression'
manifest = json.loads((out / 'manifest.json').read_text())
summary = json.loads((out / 'summary.json').read_text())
print(manifest['status'], manifest.get('heartbeat_at'), manifest['random_stage'])
for r in summary['rows']:
    c, t = r.get('counts', {}), r.get('timing_summary', {})
    print(r['date'], r['market'], r['status'], c.get('matched'), c.get('mismatched'), c.get('excluded_by_status'), c.get('data_errors'), c.get('missing_source'), t.get('restore_total_seconds'), t.get('validation_total_seconds'), r.get('elapsed_seconds'), r.get('peak_rss_kib'))

In [ ]:
for r in summary['rows']:
    if r.get('first_anomalies') or r.get('error'):
        print(r['date'], r['market'], r.get('first_anomalies'), r.get('error'))
# 完整失败帧在逐日 full.json；此处只展示每个任务最多 20 条定位样例。

## 计时口径与证据边界
恢复 = 输入读取/分片 + 扣除验证回调的回放 + 分片清理；对比 = 参考帧读取/分类 + 验证回调 + 报告整理。总时间另含进程启动与序列化输出。并发下的 wall time 会受 I/O/调度影响，不能当作 CPU 时间或单独运行纯回放的耗时。GNU time 记录峰值 RSS。
脚本保留输入行数、schema、大小和 mtime，并核对输入未被替换；这不是完整输入内容哈希。源代码与二进制使用 SHA256。成功运行删除临时分片，失败保留其位置。若出现任何失败，不改变比较窗口或恢复算法，先输出首个差异字段及事件序号供诊断。